# Train FPL Points Prediction Model
This notebook focuses on training a model to predict FPL points using the features from the previous notebooks.

## 1. Include required libraries

In [ ]:
import pandas as pd
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import joblib

## 2. Read Data
We read the CSV files generated by earlier steps into Spark dataframes:
* features/XXX.csv

In [ ]:
# Load the dataset
data = pd.read_csv('part-00000-8cbcc889-0182-4e99-9ae0-cf8103c46ad9-c000.csv')

your 131072x1 screen size is bogus. expect trouble
25/03/23 10:56:04 WARN Utils: Your hostname, LAPTOP-3B4JAVBH resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/03/23 10:56:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/23 10:56:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 3. Data Cleaning
We'll handle data cleaning tasks such as dealing with missing values.

In [ ]:
# Drop rows with missing values
data = data.dropna()

# Verify no missing values
print("Missing values after handling:\n", data.isnull().sum())

+--------------------+--------------+--------+-------------------+---+------------+--------------+-------+----------------+--------------------------+--------------+-----------------------+------------+----------------+---------------+-----+---+-------+------------+---------+---------+------+--------+---------+----------+------+---------+------------+
|                name|          team|position|       kickoff_time| GW|goals_scored|goals_conceded|assists|expected_assists|expected_goal_involvements|expected_goals|expected_goals_conceded|total_points|penalties_missed|penalties_saved|saves|bps|minutes|yellow_cards|red_cards|own_goals|starts|was_home|influence|creativity|threat|ict_index|clean_sheets|
+--------------------+--------------+--------+-------------------+---+------------+--------------+-------+----------------+--------------------------+--------------+-----------------------+------------+----------------+---------------+-----+---+-------+------------+---------+---------+------

## 4. Feature Engineering
We'll do one-hot encoding on categorical features.

### Features
* was_home_true
* was_home_false

In [ ]:
# Perform one-hot encoding on the 'was_home' column
data = pd.get_dummies(data, columns=['was_home'], drop_first=True)

+---------------+--------+--------+-------------------+---+------------+--------------+-------+----------------+--------------------------+--------------+-----------------------+------------+----------------+---------------+-----+---+-------+------------+---------+---------+------+--------+---------+----------+------+---------+------------+-------------------+----------------------+---------------------+------------------------+--------------+-----------------+-----------------------+--------------------------+---------------------------------+------------------------------------+---------------------+------------------------+------------------------------+---------------------------------+-------------------+----------------------+-----------------------+--------------------------+----------------------+-------------------------+------------+---------------+----------+-------------+--------------+-----------------+-------------------+----------------------+----------------+-----------

## 5. Train Model
We'll train the model.

In [ ]:
# Prepare the data for the model
X = data.drop('total_points', axis=1)
y = data['total_points']

# Define cross-validation parameters
cv = KFold(n_splits=10, shuffle=True, random_state=42)

# Initialize models
linear_model = LinearRegression()
rf_model = RandomForestRegressor(random_state=42)

# Perform cross-validation for Linear Regression
linear_mse_scores = -cross_val_score(linear_model, X, y, cv=cv, scoring='neg_mean_squared_error')
linear_r2_scores = cross_val_score(linear_model, X, y, cv=cv, scoring='r2')

# Perform cross-validation for Random Forest Regression
rf_mse_scores = -cross_val_score(rf_model, X, y, cv=cv, scoring='neg_mean_squared_error')
rf_r2_scores = cross_val_score(rf_model, X, y, cv=cv, scoring='r2')

# Print the cross-validation results
print("Linear Regression Cross-Validation Results:")
print(f"Average MSE: {np.mean(linear_mse_scores)}")
print(f"Average R-squared: {np.mean(linear_r2_scores)}")

print("\nRandom Forest Regression Cross-Validation Results:")
print(f"Average MSE: {np.mean(rf_mse_scores)}")
print(f"Average R-squared: {np.mean(rf_r2_scores)}")

# Retrain the best model on the entire dataset
# For this example, let's assume Random Forest performed better
best_model = RandomForestRegressor(random_state=42)
best_model.fit(X, y)

## 6. Write Data
Finally, we'll write the transformed data to CSV files for use in the subsequent model training notebook.

In [ ]:
from datetime import datetime
import os

# Define output directory
# We define the directory where the processed data will be saved.
data_source = "models"
current_datetime = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"../data/processed/{data_source}/{current_datetime}"
filename = 'model.joblib'
filepath = os.path.join(output_dir, filename)

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Create the directory if it doesn't exist
# We create the directory if it doesn't exist.
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Save the model
joblib.dump(best_model, filepath)

print(f"Model training complete. Model saved to {output_dir}")

ETL process complete. Transformed data saved to ../data/processed/features/20250322_212823
